<a href="https://colab.research.google.com/github/kimdesok/Graphcore_IPU/blob/main/batch_resnet_performance_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, time, gc
import numpy as np
import pandas as pd
import torch
#import poptorch
import torchvision.models as models

from datasets import load_dataset, load_from_disk
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

import warnings
# Hide the specific PyTorch positional args warning
warnings.filterwarnings("ignore", category=UserWarning, message="positional args are being deprecated")

In [15]:
DRIVE_PATH = "./drive/MyDrive/camelyon17-wilds"
my_dataset = "wltjr1007/Camelyon17-WILDS"

train_path = os.path.join(DRIVE_PATH, "datasets/train")
dataset_path = os.path.join(DRIVE_PATH, "datasets")
model_path = os.path.join(DRIVE_PATH, "models")
results_path = os.path.join(DRIVE_PATH, "results/resnet")


os.makedirs(model_path, exist_ok=True)
os.makedirs(results_path, exist_ok=True)

CSV_FILE = os.path.join(results_path, "camelyon17_gpu_benchmark.csv")

In [16]:
print(CSV_FILE)

./drive/MyDrive/camelyon17-wilds/results/resnet/camelyon17_gpu_benchmark.csv


## Dataset / transform / adapter

In [4]:
class Camelyon17Dataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None, return_metadata=False):
        self.hf_dataset = hf_dataset
        self.transform = transform
        self.return_metadata = return_metadata

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        sample = self.hf_dataset[idx]
        image = sample["image"].convert("RGB")
        label = torch.tensor(sample["label"], dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        if self.return_metadata:
            metadata = {k: sample[k] for k in [
                "center", "image_id", "patient", "node",
                "x_coord", "y_coord", "slide"
            ]}
            return image, label, metadata

        return image, label


my_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])


def load_camelyon17():

    if os.path.exists(train_path) and os.listdir(train_path):
        print("Loading dataset from local disk...")
        return load_from_disk(dataset_path)

    print("Downloading dataset...")
    ds = load_dataset(my_dataset, cache_dir=dataset_path)
    ds.save_to_disk(dataset_path)
    return ds

def make_random_subset(hf_dataset, n_samples=None, seed=42):
    shuffled = hf_dataset.shuffle(seed=seed)
    if n_samples is None:
        return shuffled
    return shuffled.select(range(min(n_samples, len(shuffled))))


def make_balanced_subset(hf_dataset, n_per_class=5000, seed=42, label_key="label"):
    shuffled = hf_dataset.shuffle(seed=seed)
    selected, indices = {}, []

    for i, label in enumerate(shuffled[label_key]):
        label = int(label)
        selected.setdefault(label, 0)

        if selected[label] < n_per_class:
            indices.append(i)
            selected[label] += 1

        if len(selected) >= 2 and all(v >= n_per_class for v in selected.values()):
            break

    return shuffled.select(indices)

##  Model factory

In [5]:
class ResNetCamelyon(torch.nn.Module):
    def __init__(self, model_type="ResNet18", num_classes=2, pretrained=True):
        super().__init__()

        if model_type == "ResNet18":
            weights = models.ResNet18_Weights.DEFAULT if pretrained else None
            self.backbone = models.resnet18(weights=weights)

        elif model_type == "ResNet34":
            weights = models.ResNet34_Weights.DEFAULT if pretrained else None
            self.backbone = models.resnet34(weights=weights)

        elif model_type == "ResNet50":
            weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
            self.backbone = models.resnet50(weights=weights)

        else:
            raise ValueError(f"Unknown model_type: {model_type}")

        in_features = self.backbone.fc.in_features
        self.backbone.fc = torch.nn.Linear(in_features, num_classes)

    def forward(self, x, labels=None):
        logits = self.backbone(x)

        if labels is None:
            return logits

        loss = torch.nn.functional.cross_entropy(logits, labels)
        return logits, loss

## Evaluation

In [6]:
#_________________________
# GPU equivalent functions
#_________________________

import os
import time
import torch
import pandas as pd
from torch.utils.data import DataLoader
from collections import Counter


def evaluate_gpu(model, val_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    n_samples = 0
    correct = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).long().view(-1)

            logits = model(images)

            if isinstance(logits, tuple):
                logits = logits[0]

            loss = criterion(logits, labels)

            preds = torch.argmax(logits, dim=1)

            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            n_samples += batch_size

            correct += (preds == labels).sum().item()

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return (
        running_loss / n_samples,
        correct / n_samples,
        Counter(all_preds),
        Counter(all_labels),
    )

def test_gpu(
    model_type,
    checkpoint,
    test_hf,
    transform,
    device,
    batch_size=64,
    num_workers=4,
):
    test_data = Camelyon17Dataset(test_hf, transform=transform)

    test_loader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    model = ResNetCamelyon(model_type=model_type, pretrained=False)
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    model.to(device)
    model.eval()

    criterion = torch.nn.CrossEntropyLoss()

    test_loss, test_acc, pred_counter, label_counter = evaluate_gpu(
        model=model,
        val_loader=test_loader,
        criterion=criterion,
        device=device,
    )

    print("preds", pred_counter)
    print("labels", label_counter)
    print("test_accuracy =", test_acc)

    return {
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_pred_0": pred_counter.get(0, 0),
        "test_pred_1": pred_counter.get(1, 0),
        "test_label_0": label_counter.get(0, 0),
        "test_label_1": label_counter.get(1, 0),
    }

## Experiment Function

In [7]:
#__________________________
# GPU equivalent function

def run_experiment_gpu(config, dataset):
    model_type = config["model_type"]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

    # ------------------------------------------------------------------
    # Data split
    # ------------------------------------------------------------------
    if config["whole_data"]:
        train_hf = dataset["train"]
        val_hf = dataset["validation"]
        test_hf = dataset["test"]
    else:
        if config["balanced"]:
            train_hf = make_balanced_subset(
                dataset["train"],
                config["n_train_per_class"],
                seed=42,
            )
            val_hf = make_balanced_subset(
                dataset["validation"],
                config["n_val_per_class"],
                seed=43,
            )
            test_hf = make_balanced_subset(
                dataset["test"],
                config["n_test_per_class"],
                seed=44,
            )
        else:
            train_hf = make_random_subset(dataset["train"], config["n_train"], seed=42)
            val_hf = make_random_subset(dataset["validation"], config["n_val"], seed=43)
            test_hf = make_random_subset(dataset["test"], config["n_test"], seed=44)

    # ------------------------------------------------------------------
    # Save file
    # ------------------------------------------------------------------
    save_file = (
        f"{model_path}/{config['model_type']}_"
        f"train{len(train_hf)}_"
        f"gpu_"
        f"lr{config['lr']}_"
        f"best.pt"
    )

    print("Save file:", save_file)

    # ------------------------------------------------------------------
    # Dataset / DataLoader
    # ------------------------------------------------------------------
    train_data = Camelyon17Dataset(train_hf, transform=my_transform)
    val_data = Camelyon17Dataset(val_hf, transform=my_transform)

    train_loader = DataLoader(
        train_data,
        batch_size=config["batch_size"],
        shuffle=True,
        drop_last=True,
        num_workers=4,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_data,
        batch_size=config.get("val_batch_size", 64),
        shuffle=False,
        drop_last=False,
        num_workers=4,
        pin_memory=True,
    )

    # ------------------------------------------------------------------
    # Model / optimizer / loss
    # ------------------------------------------------------------------
    model = ResNetCamelyon(model_type=model_type, pretrained=True)
    model.to(device)

    criterion = torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    best_val_loss = float("inf")
    best_val_acc = 0.0
    current_lr = config["lr"]

    epochs_without_improvement = 0
    epochs_without_lr_improvement = 0

    start_train = time.time()

    # ------------------------------------------------------------------
    # Training loop
    # ------------------------------------------------------------------
    for epoch in range(config["num_epochs"]):
        t0 = time.time()

        model.train()

        running_loss = 0.0
        n_samples = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(images)

            # If model returns (logits, loss), keep only logits
            if isinstance(logits, tuple):
                logits = logits[0]

            logits = logits.view(-1)
            labels_float = labels.float().view(-1)

            #print(f"Logits shape: {logits.shape}, Labels shape: {labels_float.shape}")

            logits = model(images)

            if isinstance(logits, tuple):
                logits = logits[0]

            labels = labels.to(device, non_blocking=True).long().view(-1)

            #print(f"Logits shape: {logits.shape}, Labels shape: {labels.shape}")
            #print("labels dtype:", labels.dtype)

            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            n_samples += batch_size

        train_loss = running_loss / n_samples

        val_loss, val_acc, val_pred_counter, val_label_counter = evaluate_gpu(
            model=model,
            val_loader=val_loader,
            criterion=criterion,
            device=device,
        )

        print(
            f"\n{model_type} GPU | epoch {epoch+1}/{config['num_epochs']} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"lr={current_lr:.2e} | "
            f"time={time.time()-t0:.1f}s"
        )

        print("val preds :", val_pred_counter)
        print("val labels:", val_label_counter)

        # Same rule as IPU version: checkpoint by val_acc
        if val_acc > best_val_acc:
            best_val_loss = val_loss
            best_val_acc = val_acc
            epochs_without_improvement = 0
            epochs_without_lr_improvement = 0

            torch.save(model.state_dict(), save_file)

            print(
                "\N{WHITE HEAVY CHECK MARK} Saved best model:",
                save_file,
                "val_acc:",
                val_acc,
            )
        else:
            epochs_without_improvement += 1
            epochs_without_lr_improvement += 1

        # Manual ReduceLROnPlateau-like behavior
        if epochs_without_lr_improvement >= config["lr_patience"]:
            new_lr = max(current_lr * config["lr_factor"], config["min_lr"])

            if new_lr < current_lr:
                current_lr = new_lr
                print(f"Reducing LR to {current_lr:.2e}")

                for group in optimizer.param_groups:
                    group["lr"] = current_lr

                epochs_without_lr_improvement = 0

        if epochs_without_improvement >= config["patience"]:
            print("Early stopping triggered.")
            break

    train_time_min = (time.time() - start_train) / 60

    # ------------------------------------------------------------------
    # Test best checkpoint
    # ------------------------------------------------------------------
    test_metrics = test_gpu(
        model_type=model_type,
        checkpoint=save_file,
        test_hf=test_hf,
        transform=my_transform,
        device=device,
        batch_size=config.get("test_batch_size", 64),
        num_workers=4,
    )

    result = {
        "backend": "gpu",
        "model_type": model_type,
        "whole_data": config["whole_data"],
        "balanced": config["balanced"],
        "train_size": len(train_hf),
        "val_size": len(val_hf),
        "test_size": len(test_hf),
        "batch_size": config["batch_size"],
        "lr": config["lr"],
        "weight_decay": config["weight_decay"],
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "train_time_min": train_time_min,
        "checkpoint": save_file,
        **test_metrics,
    }

    return result

## Preparation for multiple experiments

In [18]:
WHOLE_DATA = True
LEARN_RATE = 1e-5
LEARN_FACTOR = 0.5
EPOCHS = 30
NO_TRAIN = 50000
NO_TEST = 5000
DEVICE_ITER = 9
BATCH_SIZE = 32

resnet18_config = {
        "model_type": "ResNet18",
        "whole_data": WHOLE_DATA,
        "balanced": False,
        "n_train": 40000,
        "n_val": 4000,
        "n_test": 4000,
        "batch_size": BATCH_SIZE,
        "rep_fact": 4,
        "device_itr": DEVICE_ITER,
        "grad_accum": 2,
        "lr": LEARN_RATE,
        "weight_decay": 1e-4,
        "num_epochs":EPOCHS,
        "patience": 10,
        "lr_patience": 3,
        "lr_factor": LEARN_FACTOR,
        "min_lr": 1e-7,
        "save_file": results_path, #dummy
    }

resnet34_config = {
        "model_type": "ResNet34",
        "whole_data": WHOLE_DATA,
        "balanced": False,
        "n_train": 80000,
        "n_val": 8000,
        "n_test": 8000,
        "batch_size": BATCH_SIZE,
        "rep_fact": 4,
        "device_itr": DEVICE_ITER,
        "grad_accum": 2,
        "lr": LEARN_RATE,
        "weight_decay": 1e-4,
        "num_epochs": EPOCHS,
        "patience": 10,
        "lr_patience": 3,
        "lr_factor": LEARN_FACTOR,
        "min_lr": 1e-7,
        "save_file": results_path, #dummy
    }

resnet50_config = {
        "model_type": "ResNet50",
        "whole_data": WHOLE_DATA,
        "balanced": False,
        "n_train": NO_TRAIN,
        "n_val": NO_TEST,
        "n_test": NO_TEST,
        "batch_size": BATCH_SIZE,
        "rep_fact": 4,
        "device_itr": DEVICE_ITER,
        "grad_accum": 1,
        "lr": LEARN_RATE,
        "weight_decay": 1e-4,
        "num_epochs": EPOCHS,
        "patience": 10,
        "lr_patience": 3,
        "lr_factor": LEARN_FACTOR,
        "min_lr": 1e-7,
        "save_file": results_path,  #dummy
    }

## Setting configurations for multiple exps.

In [19]:
import copy
import itertools
import traceback
from datetime import datetime

def make_config_grid(base_configs, search_space):
    configs = []

    keys = list(search_space.keys())
    values = list(search_space.values())

    for base in base_configs:
        for combo in itertools.product(*values):
            cfg = copy.deepcopy(base)

            for k, v in zip(keys, combo):
                cfg[k] = v

            cfg["run_id"] = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            cfg["save_file"] = (
                f"./models/{cfg['model_type']}_"
                f"train{cfg.get('n_train', 'full')}_"
                f"rep{cfg['rep_fact']}_"
                f"di{cfg['device_itr']}_"
                f"ga{cfg['grad_accum']}_"
                f"lr{cfg['lr']}_"
                f"{cfg['run_id']}_best.pt"
            )

            configs.append(cfg)

    return configs

## Running in the for loop with multiple models

In [ ]:
AI_ACCEL = "GPU"

dataset = load_camelyon17()

base_configs  = [
    resnet18_config,
    resnet34_config,
    resnet50_config,
]

search_space = {
    "lr": [1e-5],
    "weight_decay": [1e-4],
    "patience": [10],
    "lr_patience": [3],
}

experiments = make_config_grid(base_configs, search_space)
print(f"Number of {AI_ACCEL} experiments:", len(experiments))

results = []

for i, config in enumerate(experiments):
    print("\n" + "=" * 90)
    print(f'{AI_ACCEL} Experiment {i+1}/{len(experiments)} : {config["model_type"]}')
    print(config)
    print("=" * 90)


    result = run_experiment_gpu(config, dataset)

    pd.DataFrame([result]).to_csv(
        CSV_FILE,
        mode="a",
        header=not os.path.exists(CSV_FILE),
        index=False
    )

    print("Logged:", CSV_FILE)

Loading dataset from local disk...
Number of GPU experiments: 3

GPU Experiment 1/3 : ResNet18
{'model_type': 'ResNet18', 'whole_data': True, 'balanced': False, 'n_train': 40000, 'n_val': 4000, 'n_test': 4000, 'batch_size': 32, 'rep_fact': 4, 'device_itr': 9, 'grad_accum': 2, 'lr': 1e-05, 'weight_decay': 0.0001, 'num_epochs': 30, 'patience': 10, 'lr_patience': 3, 'lr_factor': 0.5, 'min_lr': 1e-07, 'save_file': './models/ResNet18_train40000_rep4_di9_ga2_lr1e-05_20260628_005828_667575_best.pt', 'run_id': '20260628_005828_667575'}
Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Save file: ./drive/MyDrive/camelyon17-wilds/models/ResNet18_train302436_gpu_lr1e-05_best.pt

ResNet18 GPU | epoch 1/30 | train_loss=0.0571 | val_loss=0.1258 | val_acc=0.9536 | lr=1.00e-05 | time=143.0s
val preds : Counter({1: 35226, 0: 33238})
val labels: Counter({0: 34404, 1: 34060})
✅ Saved best model: ./drive/MyDrive/camelyon17-wilds/models/ResNet18_train302436_gpu_lr1e-05_best.pt val_acc: 0

100%|██████████| 83.3M/83.3M [00:00<00:00, 379MB/s]



ResNet34 GPU | epoch 1/30 | train_loss=0.0502 | val_loss=0.1190 | val_acc=0.9598 | lr=1.00e-05 | time=210.2s
val preds : Counter({1: 35224, 0: 33240})
val labels: Counter({0: 34404, 1: 34060})
✅ Saved best model: ./drive/MyDrive/camelyon17-wilds/models/ResNet34_train302436_gpu_lr1e-05_best.pt val_acc: 0.9597744800186959

ResNet34 GPU | epoch 2/30 | train_loss=0.0167 | val_loss=0.1212 | val_acc=0.9600 | lr=1.00e-05 | time=210.0s
val preds : Counter({1: 35424, 0: 33040})
val labels: Counter({0: 34404, 1: 34060})
✅ Saved best model: ./drive/MyDrive/camelyon17-wilds/models/ResNet34_train302436_gpu_lr1e-05_best.pt val_acc: 0.9600373919139986

ResNet34 GPU | epoch 3/30 | train_loss=0.0079 | val_loss=0.1823 | val_acc=0.9532 | lr=1.00e-05 | time=210.1s
val preds : Counter({1: 36209, 0: 32255})
val labels: Counter({0: 34404, 1: 34060})

ResNet34 GPU | epoch 4/30 | train_loss=0.0047 | val_loss=0.2088 | val_acc=0.9518 | lr=1.00e-05 | time=210.0s
val preds : Counter({1: 35892, 0: 32572})
val labe

In [1]:
from google.colab import runtime

# This immediately terminates the current runtime environment
runtime.unassign()